# Feature Engineering Notebook
**Project:** NexaChain Intelligence Platform
**Phase:** Week 2 (Machine Learning Prep)

This notebook applies feature engineering techniques (Lag Variables, Rolling Averages, Mutual Information) to prepare the datasets for predictive modeling.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import mutual_info_regression

cleaned_dir = r'../Data/cleaned'
fe_dir = r'../Data/feature_engineered'
os.makedirs(fe_dir, exist_ok=True)

orders = pd.read_csv(os.path.join(cleaned_dir, 'orders_cleaned.csv'))
print(f'Loaded {len(orders)} orders.')

## 1. Time-Series Feature Engineering (Lag & Rolling)
We need to engineer rolling features for continuous risk tracking.

In [ ]:
# Convert dates
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders = orders.sort_values(by='order_date')

# Rolling Average of Order Value (7-day window per product)
if 'product_id' in orders.columns:
    orders['rolling_7d_order_val'] = orders.groupby('product_id')['order_value_usd'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
    
# Lag Feature: Previous Delivery Delay
if 'delivery_delay_days' in orders.columns:
    orders['prev_delivery_delay'] = orders.groupby('vendor_id')['delivery_delay_days'].shift(1)
    orders['prev_delivery_delay'] = orders['prev_delivery_delay'].fillna(0)

orders.to_csv(os.path.join(fe_dir, 'orders_fe.csv'), index=False)
print('Feature engineered orders dataset saved.')